In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', '8.png', '2.png', '19.png', '18.png', '13.png', '20.png', '11.png', '3.png', '14.png', '9.png', '1.png', '17.png', '6.png', '16.png', '4.png', '15.png', '12.png', '10.png', '7.png', '5.png', '368651_Byzantine_monograms']


In [ ]:
import os

# Chercher dans tous les emplacements possibles
search_paths = [
    '/content/drive/MyDrive/368651_Byzantine_monograms/runs',
    '/content/byzantine/runs',
    '/content/runs',
    '/content/drive/MyDrive/runs'
]

for path in search_paths:
    if os.path.exists(path):
        print(f"✓ Dossier trouvé : {path}")
        for fold in range(5):
            weight_path = os.path.join(path, f'fold_{fold}', 'weights', 'best.pt')
            if os.path.exists(weight_path):
                print(f"  → Fold {fold} : OK")
            else:
                print(f"  → Fold {fold} : MANQUANT")
    else:
        print(f"✗ Dossier introuvable : {path}")

✓ Dossier trouvé : /content/drive/MyDrive/368651_Byzantine_monograms/runs
  → Fold 0 : OK
  → Fold 1 : OK
  → Fold 2 : OK
  → Fold 3 : OK
  → Fold 4 : OK
✗ Dossier introuvable : /content/byzantine/runs
✗ Dossier introuvable : /content/runs
✗ Dossier introuvable : /content/drive/MyDrive/runs


In [6]:

import os
import json
import csv
import numpy as np
from ultralytics import YOLO

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
BASE_DIR = '/content/drive/MyDrive/368651_Byzantine_monograms'
model_path = os.path.join(BASE_DIR, 'runs', f'fold_{fold}', 'weights', 'best.pt')
RUNS_DIR   = os.path.join(BASE_DIR, 'runs')
SPLITS_DIR = os.path.join(BASE_DIR, 'splits')
OUTPUT_DIR = os.path.join(BASE_DIR, 'eval_results')
N_SPLITS   = 5
IMG_SIZE   = 640
DEVICE     = 0


def get_best_weights(fold_idx):
    """Retourne le chemin vers best.pt du fold donné."""
    path = os.path.join(RUNS_DIR, f'fold_{fold_idx}', 'weights', 'best.pt')
    if not os.path.exists(path):
        raise FileNotFoundError(f"Poids introuvables : {path}")
    return path


def get_fold_yaml(fold_idx):
    """Retourne le yaml temporaire du fold (doit exister après train.py)."""
    # Si le yaml temporaire n'existe plus, on le recrée depuis les splits
    yaml_path = os.path.join(BASE_DIR, f'data_fold_{fold_idx}.yaml')
    if not os.path.exists(yaml_path):
        # Recrée le yaml minimal à partir des splits
        import yaml, shutil

        def copy_split(split):
            split_file = os.path.join(SPLITS_DIR, f'fold_{fold_idx}',
                                      'train.txt' if split == 'train' else 'test.txt')
            img_dir = os.path.join(BASE_DIR, f'_tmp_eval_{fold_idx}', split, 'images')
            lbl_dir = os.path.join(BASE_DIR, f'_tmp_eval_{fold_idx}', split, 'labels')
            os.makedirs(img_dir, exist_ok=True)
            os.makedirs(lbl_dir, exist_ok=True)

            with open(split_file, 'r') as f:
                paths = [l.strip() for l in f if l.strip()]

            for img_path in paths:
                if not os.path.isabs(img_path):
                    img_path = os.path.join(BASE_DIR, img_path)
                fname = os.path.basename(img_path)
                stem  = os.path.splitext(fname)[0]
                dst_img = os.path.join(img_dir, fname)
                if not os.path.exists(dst_img):
                    shutil.copy2(img_path, dst_img)
                lbl_src = os.path.join(BASE_DIR, 'labels', 'train', stem + '.txt')
                dst_lbl = os.path.join(lbl_dir, stem + '.txt')
                if os.path.exists(lbl_src) and not os.path.exists(dst_lbl):
                    shutil.copy2(lbl_src, dst_lbl)

            return os.path.join(BASE_DIR, f'_tmp_eval_{fold_idx}', split)

        train_dir = copy_split('train')
        val_dir   = copy_split('val')

        # Charge nc et names depuis data_config.yaml
        with open(os.path.join(BASE_DIR, 'data_config.yaml'), 'r', encoding='utf-8') as f:
            base = yaml.safe_load(f)

        config = {'train': train_dir, 'val': val_dir,
                  'nc': base['nc'], 'names': base['names']}
        with open(yaml_path, 'w', encoding='utf-8') as f:
            yaml.dump(config, f, allow_unicode=True)

    return yaml_path


def evaluate_fold(fold_idx):
    print(f"\n{'='*60}")
    print(f"  ÉVALUATION FOLD {fold_idx + 1} / {N_SPLITS}")
    print(f"{'='*60}\n")

    weights  = get_best_weights(fold_idx)
    yaml_path = get_fold_yaml(fold_idx)

    model   = YOLO(weights)
    metrics = model.val(
        data    = yaml_path,
        imgsz   = IMG_SIZE,
        device  = DEVICE,
        verbose = True,
    )

    results = {
        'fold':       fold_idx,
        'mAP50':      round(float(metrics.box.map50), 4),
        'mAP50_95':   round(float(metrics.box.map),   4),
        'precision':  round(float(metrics.box.mp),    4),
        'recall':     round(float(metrics.box.mr),    4),
    }

    print(f"\n  Résultats fold {fold_idx} :")
    for k, v in results.items():
        if k != 'fold':
            print(f"    {k:12s} : {v}")

    return results


def aggregate_results(all_results):
    """Calcule moyenne et écart-type sur les 5 folds."""
    metrics = ['mAP50', 'mAP50_95', 'precision', 'recall']
    summary = {}
    for m in metrics:
        values = [r[m] for r in all_results]
        summary[m] = {
            'mean': round(float(np.mean(values)), 4),
            'std':  round(float(np.std(values)),  4),
            'min':  round(float(np.min(values)),  4),
            'max':  round(float(np.max(values)),  4),
        }
    return summary


def save_results(all_results, summary):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── CSV ──────────────────────────────────────
    csv_path = os.path.join(OUTPUT_DIR, 'results_per_fold.csv')
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['fold', 'mAP50', 'mAP50_95', 'precision', 'recall'])
        writer.writeheader()
        writer.writerows(all_results)

        # Ligne résumé
        writer.writerow({})
        writer.writerow({'fold': 'mean',
                         'mAP50':     summary['mAP50']['mean'],
                         'mAP50_95':  summary['mAP50_95']['mean'],
                         'precision': summary['precision']['mean'],
                         'recall':    summary['recall']['mean']})
        writer.writerow({'fold': 'std',
                         'mAP50':     summary['mAP50']['std'],
                         'mAP50_95':  summary['mAP50_95']['std'],
                         'precision': summary['precision']['std'],
                         'recall':    summary['recall']['std']})

    print(f"\n  → CSV sauvegardé : {csv_path}")

    # ── JSON ─────────────────────────────────────
    json_path = os.path.join(OUTPUT_DIR, 'results_summary.json')
    output = {'per_fold': all_results, 'summary': summary}
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"  → JSON sauvegardé : {json_path}")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    all_results = []

    for fold_idx in range(N_SPLITS):
        try:
            result = evaluate_fold(fold_idx)
            all_results.append(result)
        except FileNotFoundError as e:
            print(f"    Fold {fold_idx} ignoré : {e}")

    if not all_results:
        print("Aucun fold évalué. Vérifiez que l'entraînement est terminé.")
        return

    print(f"\n{'='*60}")
    print("  RÉSUMÉ GLOBAL")
    print(f"{'='*60}")

    summary = aggregate_results(all_results)
    for metric, stats in summary.items():
        print(f"  {metric:12s} : {stats['mean']:.4f} ± {stats['std']:.4f}  "
              f"[{stats['min']:.4f} – {stats['max']:.4f}]")

    save_results(all_results, summary)
    print("\n✓ Évaluation terminée !")


if __name__ == "__main__":
    main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

  ÉVALUATION FOLD 1 / 5

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,144,547 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 1.7±2.3 ms, read: 0.3±0.1 MB/s, size: 136.9 KB)
val: Scanning /content/drive/MyDrive/368651_Byzantine_monograms/_tmp_eval_0/val/labels... 71 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 71/71 1.2it/s 58.5s
val: New cache created: /content/drive/MyDrive/368651_Byzantine_monograms/_tmp_eval_0/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.6it/s 3.1s
                   all   